# Data Description
- Background: 台北市 YouBike 2.0 原始交易資料，用於分析 NTU 區域的 YouBike 使用情況。資料包含每次租借交易的完整紀錄，包括借還車時間、站點、租借時長、車輛類型等資訊。
- date received: 202403-202501（持續更新）
- Path to data file: `data/raw/previews/YYYYMM_YouBike_preview.csv`（預覽資料），完整資料路徑：`data/raw/YYYYMM_YouBike.csv`
- Unit of observation: 單次 YouBike 租借交易紀錄
- Sample period: 2024年3月 - 2025年1月
- Known issues:
    - 原始資料檔案較大（每個月約200MB+），預覽檔僅為樣本
    - 部分站點名稱可能包含特殊字元或格式不一致
    - 租借時長為時間格式（HH:MM:SS），需轉換為數值才能進行統計分析
- Definition for each variable: 
    - borrow_datetime: 借車時間（日期時間格式，YYYY-MM-DD HH:MM:SS）
    - borrow_site: 借車站點名稱
    - return_datetime: 還車時間（日期時間格式，YYYY-MM-DD HH:MM:SS）
    - return_site: 還車站點名稱
    - borrowing_time: 租借時長（時間格式，HH:MM:SS）
    - type: 車輛類型（一般車/電輔車）
    - date: 交易日期（YYYY-MM-DD） 


In [1]:
import pandas as pd
import glob
import os

## 資料設定

In [2]:
# 選擇要分析的資料檔案（可選擇單一月份或合併所有月份）
# 單一月份範例（使用預覽檔）：
input_data_file = "./previews/202501_YouBike_preview.csv"

# 或合併所有月份預覽資料（取消註解以使用）：
# preview_files = glob.glob("./previews/*_preview.csv")
# preview_files.sort()
# input_data_file = preview_files  # 將讀取多個檔案

# 如需使用完整原始資料（檔案較大），可取消註解以下程式碼：
# input_data_file = "./202501_YouBike.csv"

## Summary

In [3]:
# 讀取資料
if isinstance(input_data_file, list):
    # 如果 input_data_file 是列表，合併所有檔案
    df_list = []
    for file in input_data_file:
        df_temp = pd.read_csv(file)
        df_list.append(df_temp)
    df = pd.concat(df_list, ignore_index=True)
else:
    # 讀取單一檔案
    df = pd.read_csv(input_data_file)

print(f"資料形狀: {df.shape}")
print(f"欄位: {list(df.columns)}")
print(f"\n資料筆數: {len(df)}")

資料形狀: (20, 7)
欄位: ['borrow_datetime', 'borrow_site', 'return_datetime', 'return_site', 'borrowing_time', 'type', 'date']

資料筆數: 20


### Sample

In [4]:
df.head(10)

,borrow_datetime,borrow_site,return_datetime,return_site,borrowing_time,type,date
0,2025-01-13 15:00:00,捷運古亭站(8號出口),2025-01-13 21:00:00,新生南路三段52號前,05:34:25,電輔車,2025-01-13
1,2025-01-13 11:00:00,內湖區農會,2025-01-13 11:00:00,宏匯瑞光廣場(港墘路),00:10:19,一般車,2025-01-13
2,2025-01-13 01:00:00,辛亥路一段30號前,2025-01-13 03:00:00,捷運公館站(4號出口),01:50:15,一般車,2025-01-13
3,2025-01-13 11:00:00,臺大土木研究大樓前,2025-01-13 12:00:00,立人國小,00:15:08,一般車,2025-01-13
4,2025-01-13 18:00:00,忠孝東路四段248巷口,2025-01-13 18:00:00,新生和平路口東北側,00:35:48,一般車,2025-01-13
5,2025-01-13 12:00:00,臺大男一舍前,2025-01-13 12:00:00,捷運公館站(2號出口),00:06:22,一般車,2025-01-13
6,2025-01-13 20:00:00,復興忠孝東路口(東南側),2025-01-13 20:00:00,成功國宅,00:12:30,一般車,2025-01-13
7,2025-01-13 17:00:00,和平泰順街口,2025-01-13 17:00:00,羅斯福路三段245號前,00:04:40,電輔車,2025-01-13
8,2025-01-13 20:00:00,捷運芝山站(2號出口)_1,2025-01-13 21:00:00,蘭雅公園,00:08:56,一般車,2025-01-13
9,2025-01-13 11:00:00,忠順區民活動中心,2025-01-13 12:00:00,忠順區民活動中心,00:08:33,電輔車,2025-01-13


### Summary Stats

In [5]:
df.describe()

,borrow_datetime,borrow_site,return_datetime,return_site,borrowing_time,type,date
count,20,20,20,20,20,20,20
unique,13,20,11,20,20,2,1
top,2025-01-13 11:00:00,捷運古亭站(8號出口),2025-01-13 21:00:00,新生南路三段52號前,05:34:25,一般車,2025-01-13
freq,3,1,3,1,1,11,20


In [6]:
# 基本資料品質檢查
print("=" * 50)
print("資料品質檢查")
print("=" * 50)

# 檢查缺失值
print("\n各欄位缺失值數量:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "無缺失值")

# 檢查資料類型
print("\n各欄位資料類型:")
print(df.dtypes)

# 檢查唯一值數量
print("\n各欄位唯一值數量:")
for col in df.columns:
    print(f"  {col}: {df[col].nunique()}")

# 檢查日期範圍
if 'date' in df.columns:
    df['date_parsed'] = pd.to_datetime(df['date'])
    print(f"\n日期範圍: {df['date_parsed'].min()} 至 {df['date_parsed'].max()}")

# 檢查車輛類型分布
if 'type' in df.columns:
    print(f"\n車輛類型分布:")
    print(df['type'].value_counts())

# 檢查時間邏輯：還車時間應該晚於借車時間
if 'borrow_datetime' in df.columns and 'return_datetime' in df.columns:
    df['borrow_dt'] = pd.to_datetime(df['borrow_datetime'])
    df['return_dt'] = pd.to_datetime(df['return_datetime'])
    invalid_time = (df['return_dt'] < df['borrow_dt']).sum()
    print(f"\n還車時間早於借車時間的記錄數: {invalid_time}")

# 檢查最常借車/還車的站點
if 'borrow_site' in df.columns:
    print(f"\n最常借車的前5個站點:")
    print(df['borrow_site'].value_counts().head())
    
if 'return_site' in df.columns:
    print(f"\n最常還車的前5個站點:")
    print(df['return_site'].value_counts().head())


資料品質檢查

各欄位缺失值數量:
無缺失值

各欄位資料類型:
borrow_datetime    object
borrow_site        object
return_datetime    object
return_site        object
borrowing_time     object
type               object
date               object
dtype: object

各欄位唯一值數量:
  borrow_datetime: 13
  borrow_site: 20
  return_datetime: 11
  return_site: 20
  borrowing_time: 20
  type: 2
  date: 1

日期範圍: 2025-01-13 00:00:00 至 2025-01-13 00:00:00

車輛類型分布:
type
一般車    11
電輔車     9
Name: count, dtype: int64

還車時間早於借車時間的記錄數: 0

最常借車的前5個站點:
borrow_site
捷運古亭站(8號出口)    1
內湖區農會          1
辛亥路一段30號前      1
臺大土木研究大樓前      1
忠孝東路四段248巷口    1
Name: count, dtype: int64

最常還車的前5個站點:
return_site
新生南路三段52號前     1
宏匯瑞光廣場(港墘路)    1
捷運公館站(4號出口)    1
立人國小           1
新生和平路口東北側      1
Name: count, dtype: int64
